# 00.5 sklearn 最小建模流程（sklearn Minimal Pipeline）

这份 notebook 的目标是建立一个非常小但完整的机器学习建模流程。  

你后面学 `PyTorch` 时，会不断用到这里的思路：  

- 测试集划分（train-test split）
- 基线模型（baseline model）
- 标准化（standardization）
- 指标评估（evaluation metrics）
- 数据泄漏（data leakage）

## 学习目标

学完后你应该能

1. 用 `train_test_split` 切分数据
2. 解释为什么要先切分再标准化
3. 训练一个基础分类模型
4. 用准确率和混淆矩阵评估模型
5. 理解基线模型的作用
6. 把这一流程迁移到后面的神经网络任务

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. 读入数据

这里使用 `iris` 数据集，因为它小、稳定、适合演示。  


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("特征前几行 / feature head:")
print(X.head())
print()
print("标签前几行 / target head:")
print(y.head())
print()
print("类别名称 / target names:", iris.target_names)

## 2. 训练集与测试集

为什么要切分？  

- training set: 用来学习参数
- test set: 用来评估泛化（used to estimate generalization）

如果你在测试集上调来调去，评估就会失真。  


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. 基线模型

baseline model 的意义是：先有一个最低参考线。  

如果你的复杂模型还不如基线模型，那通常说明流程有问题。  


In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_pred = dummy_clf.predict(X_test)
dummy_acc = accuracy_score(y_test, dummy_pred)

print("基线准确率 / baseline accuracy:", dummy_acc)

## 4. 标准化与 Pipeline

很多模型受特征尺度影响，所以常要标准化  

为什么建议用 `Pipeline`？  

- 把预处理和模型绑定在一起
- 降低数据泄漏风险（reduces data leakage risk）
- 代码更清晰（keeps the code cleaner）

In [ ]:
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500)),
    ]
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
acc = accuracy_score(y_test, pred)

print("逻辑回归准确率 / logistic regression accuracy:", acc)

注意顺序

1. 先切分
2. 再用训练集拟合标准化器
3. 再把同样的变换用到测试集

data leakage 的关键之一。  


## 5. 评估

accuracy，分类任务里还应关注：

- 混淆矩阵（confusion matrix）
- 分类报告（classification report）

因为单个准确率会隐藏错误结构。  


In [ ]:
cm = confusion_matrix(y_test, pred)
report = classification_report(y_test, pred, target_names=iris.target_names)

print("混淆矩阵 / confusion matrix:")
print(cm)
print()
print("分类报告 / classification report:")
print(report)

## 6. 和基线比较

真正有意义的不是某个准确率数字本身，而是它相对基线提高了多少。  


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["DummyClassifier", "LogisticRegression"],
        "accuracy": [dummy_acc, acc],
    }
)

print(comparison)

In [ ]:
# 练习 1
# 请实现一个函数 build_logreg_pipeline()。
# Implement build_logreg_pipeline().
#
# 要求
# 1. 返回一个 Pipeline
# 2. 第一层是 StandardScaler
# 3. 第二层是 LogisticRegression(max_iter=500)

def build_logreg_pipeline():
    # TODO
    pass


# test_pipe = build_logreg_pipeline()
# print(test_pipe)

In [ ]:
# 练习 1 参考答案

def build_logreg_pipeline_solution():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=500)),
        ]
    )


print(build_logreg_pipeline_solution())

In [ ]:
# 练习 2
# 用一句话回答
# 为什么标准化器不能先在全量数据上 fit，再切分训练集和测试集？
# Why should we not fit the scaler on the full dataset before splitting into train and test?

参考回答

如果标准化器先看到了全量数据，就等于让训练流程提前接触了测试集信息，这会导致数据泄漏  


## 7. 小结

这份 notebook 的关键不是某个具体模型，而是完整流程。  

你现在应该能回答

1. 为什么先切分再标准化？
2. 为什么要先做基线模型？
3. 混淆矩阵能补充什么信息？
4. `Pipeline` 为什么更安全、更清晰？

下一步建议

- `Phase 0` 基础已经形成闭环，接下来可以进入 `PyTorch` 的 `Tensor` 基础（Phase 0 now forms a closed loop, so the next natural step is `PyTorch` tensor basics.）